In [8]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ==============================
# CONFIGURACIÓN
# ==============================
DATASET_PATH = "dataset"
MODEL_PATH = "modelos/gestos_model.pkl"

os.makedirs("modelos", exist_ok=True)

# ==============================
# CARGA DE DATOS
# ==============================
data, labels = [], []
archivos = glob.glob(os.path.join(DATASET_PATH, "*.csv"))

print(f"Cargando {len(archivos)} archivos desde '{DATASET_PATH}'...\n")

for archivo in archivos:
    nombre = os.path.splitext(os.path.basename(archivo))[0]
    print(f"Cargando: {nombre}")
    df = pd.read_csv(archivo, header=None)
    df = df.dropna(axis=0, how='any')
    X = df.values.astype(float)
    y = np.array([nombre] * len(X))
    data.append(X)
    labels.append(y)

X = np.vstack(data)
y = np.concatenate(labels)

print(f"\nTotal de muestras: {len(X)} | Gestos: {set(y)}")

# ==============================
# DIVISIÓN Y ENTRENAMIENTO
# ==============================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\nEntrenando modelo RandomForest...")
modelo = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
modelo.fit(X_train, y_train)

# ==============================
# EVALUACIÓN
# ==============================
y_pred = modelo.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nPrecisión del modelo: {acc * 100:.2f}%\n")

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# ==============================
# GUARDAR MODELO
# ==============================
joblib.dump(modelo, MODEL_PATH)
print(f"Modelo guardado en '{MODEL_PATH}'")


Cargando 5 archivos desde 'dataset'...

Cargando: abrirPuerta
Cargando: anteriorMelodia
Cargando: modoNormal
Cargando: modoSuper
Cargando: siguienteMelodia

Total de muestras: 50000 | Gestos: {np.str_('anteriorMelodia'), np.str_('modoNormal'), np.str_('abrirPuerta'), np.str_('siguienteMelodia'), np.str_('modoSuper')}

Entrenando modelo RandomForest...

Precisión del modelo: 100.00%

Matriz de confusión:
[[2000    0    0    0    0]
 [   0 2000    0    0    0]
 [   0    0 2000    0    0]
 [   0    0    0 2000    0]
 [   0    0    0    0 2000]]

Reporte de clasificación:
                  precision    recall  f1-score   support

     abrirPuerta       1.00      1.00      1.00      2000
 anteriorMelodia       1.00      1.00      1.00      2000
      modoNormal       1.00      1.00      1.00      2000
       modoSuper       1.00      1.00      1.00      2000
siguienteMelodia       1.00      1.00      1.00      2000

        accuracy                           1.00     10000
       macro avg 

# Pipeline Multimodal SynkroHogar – Versión Final

Este sistema integra:

- GestureNet (MediaPipe + SVM)
- PhraseNet-AR (Generación de texto contextual según comando)
- VITS (Modelo TTS español css10)
- Comunicación con ESP32-CAM
- Reproducción de audio local
- Overlay visual de estados

## Flujo completo:
1. Se detecta un gesto y se suaviza la predicción.
2. Se inicia un temporizador de confianza.
3. Cuando el gesto es estable → genera texto con PhraseNetAR.
4. El texto se convierte en audio con VITS.
5. El audio se reproduce sin bloquear el sistema.
6. Se envía el comando al ESP32.
7. Se entra en cooldown para evitar repeticiones.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DGRNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DGRNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, output_dim)
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(128)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        return x


In [9]:
import numpy as np
label_path = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\label_encoder.npy"
clases = np.load(label_path, allow_pickle=True)
print(clases)


LabelEncoder()


In [10]:
import joblib

encoder = joblib.load(r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\label_encoder.pkl")
print(encoder.classes_)


['abrirPuerta' 'anteriorMelodia' 'apagarLuces' 'encenderLuces'
 'modoNormal' 'modoSuper' 'siguienteMelodia']


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf')


import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import joblib
import time
import threading
import requests
from collections import deque
import sys

# ==============================
# CONFIGURACIÓN GENERAL
# ==============================
ESP32_IP = "192.168.191.128"
MODEL_PATH = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\dgrnet_gestos.pt"
ENCODER_PATH = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\label_encoder.pkl"
SCALER_PATH  = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\scaler.pkl"

STREAM_URL = f"http://{ESP32_IP}:81/stream"
COMANDO_URL = f"http://{ESP32_IP}:8080/cmd?msg="

TIEMPO_CONFIANZA = 2.5
TIEMPO_COOLDOWN = 3.0
VENTANA = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# MODELO NEURONAL
# ==============================
class DGRNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

print("Cargando modelo neuronal...")
encoder = joblib.load(ENCODER_PATH)
scaler = joblib.load(SCALER_PATH)
num_classes = len(encoder.classes_)
model = DGRNet(126, num_classes).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Modelo cargado correctamente.\n")

# ==============================
# MEDIAPIPE
# ==============================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# ==============================
# ESTADO GLOBAL
# ==============================
frame_actual = None
captura_activa = True
pred_hist = deque(maxlen=VENTANA)
gesto_en_confianza = None
inicio_confianza = None
ultimo_ejecutado = None
inicio_cooldown = 0
bloqueado = False
modo_super_activo = False 

# ==============================
# COMANDOS
# ==============================
comandos = {
    "abrirPuerta": "CMD|OPEN_DOOR",
    "modoSuper": "CMD|SUPER_ON",
    "modoNormal": "CMD|SUPER_OFF",
    "siguienteMelodia": "CMD|CHANGE_MUSIC",
    "anteriorMelodia": "CMD|PREV_MUSIC",
    "encenderLuces": "CMD|LED_ON",
    "apagarLuces": "CMD|LED_OFF"
}

def enviar_comando(cmd):
    def tarea():
        try:
            requests.get(COMANDO_URL + cmd, timeout=1)
            print(f"[ENVÍO] {cmd}")
        except Exception as e:
            print(f"[ERROR envío] {e}")
    threading.Thread(target=tarea, daemon=True).start()

# ==============================
# CAPTURA DE VIDEO (HILO)
# ==============================
def hilo_captura():
    global frame_actual, captura_activa
    cap = cv2.VideoCapture(STREAM_URL)
    if not cap.isOpened():
        print("No se pudo conectar con la cámara ESP32.")
        captura_activa = False
        return
    print("Cámara conectada. Reconociendo gestos...")
    while captura_activa:
        ret, frame = cap.read()
        if ret:
            frame_actual = cv2.flip(frame, 1)
        else:
            time.sleep(0.05)
    cap.release()

threading.Thread(target=hilo_captura, daemon=True).start()

# ==============================
# LOOP PRINCIPAL
# ==============================
try:
    while True:
        if frame_actual is None:
            time.sleep(0.05)
            continue

        frame = frame_actual.copy()
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        puntos = []

        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                for lm in hand_landmarks.landmark:
                    puntos.extend([lm.x, lm.y, lm.z])
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            while len(puntos) < 126:
                puntos.append(0.0)
            if len(puntos) > 126:
                puntos = puntos[:126]

            X = scaler.transform([puntos])
            X_tensor = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            with torch.no_grad():
                logits = model(X_tensor)
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                pred = encoder.inverse_transform([np.argmax(probs)])[0]
            pred_hist.append(pred)
        else:
            pred_hist.clear()
            pred = None

        pred_suavizado = max(set(pred_hist), key=pred_hist.count) if pred_hist else None
        ahora = time.time()

        # === CONFIRMACIÓN Y CONTROL DE MODO SUPER ===
        if not bloqueado and pred_suavizado and pred_suavizado in comandos:
            # Bloquear comandos de luces cuando modoSuper está activo
            if modo_super_activo and pred_suavizado in ["encenderLuces", "apagarLuces"]:
                pass
            else:
                if gesto_en_confianza != pred_suavizado:
                    gesto_en_confianza = pred_suavizado
                    inicio_confianza = ahora
                else:
                    duracion = ahora - inicio_confianza
                    if duracion >= TIEMPO_CONFIANZA:
                        cmd = comandos[gesto_en_confianza]
                        enviar_comando(cmd)
                        ultimo_ejecutado = gesto_en_confianza
                        bloqueado = True
                        inicio_cooldown = ahora

                        # Cambiar estado del modo
                        if gesto_en_confianza == "modoSuper":
                            modo_super_activo = True
                            print(">> Modo SUPER activado: luces bloqueadas.")
                        elif gesto_en_confianza == "modoNormal":
                            modo_super_activo = False
                            print(">> Modo NORMAL activado: luces desbloqueadas.")

                        gesto_en_confianza = None
                        inicio_confianza = None
        elif bloqueado:
            if (ahora - inicio_cooldown) >= TIEMPO_COOLDOWN:
                bloqueado = False
        else:
            gesto_en_confianza = None
            inicio_confianza = None

        # === INFORMACIÓN VISUAL ===
        texto = ""
        color = (0, 255, 0)
        if gesto_en_confianza:
            restante = TIEMPO_CONFIANZA - (ahora - inicio_confianza)
            texto = f"Detectando {gesto_en_confianza} ({restante:.1f}s)"
        elif bloqueado:
            restante = TIEMPO_COOLDOWN - (ahora - inicio_cooldown)
            texto = f"Ejecutado '{ultimo_ejecutado}' Esperando {restante:.1f}s"
            color = (0, 140, 255)
        elif pred_suavizado:
            texto = f"Gesto: {pred_suavizado} | ModoSuper: {'ON' if modo_super_activo else 'OFF'}"

        if texto:
            cv2.putText(frame, texto, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        cv2.imshow("Reconocimiento de gestos - SynkroHogar", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):  # tecla "q" cierra todo
            print("\nCerrando sistema SynkroHogar IA...")
            captura_activa = False
            cv2.destroyAllWindows()
            sys.exit(0)

except KeyboardInterrupt:
    print("\nCierre manual detectado.")
    captura_activa = False
    cv2.destroyAllWindows()
    sys.exit(0)


Cargando modelo neuronal...
Modelo cargado correctamente.

Cámara conectada. Reconociendo gestos...
[ENVÍO] CMD|OPEN_DOOR
[ENVÍO] CMD|OPEN_DOOR
[ENVÍO] CMD|LED_ON
[ENVÍO] CMD|LED_OFF

Cierre manual detectado.


SystemExit: 0

a:\GeneracionImagenes\finalOmega\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf')

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import joblib
import time
import threading
import requests
from collections import deque
import sys
import json
import sentencepiece as spm
from torch.serialization import add_safe_globals
from TTS.api import TTS

# ==============================
# CONFIGURACIÓN GENERAL
# ==============================
ESP32_IP = "192.168.191.128"
MODEL_PATH = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\dgrnet_gestos.pt"
ENCODER_PATH = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\label_encoder.pkl"
SCALER_PATH  = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\modelos\scaler.pkl"

# ---- Integración PhraseNetAR ----
BASE_PHRASE = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\modelos\PhraseNet_AR"
TOKENIZER_PATH = BASE_PHRASE + "\\tokenizer.pt"
CMD2ID_PATH = BASE_PHRASE + "\\cmd2id.pt"
ID2CMD_PATH = BASE_PHRASE + "\\id2cmd.pt"
CONFIG_PATH = BASE_PHRASE + "\\config.json"
MODEL_PHRASE = BASE_PHRASE + "\\PhraseNetAR_best.pt"

# ---- Modelo de voz ----
MODEL_VITS = "tts_models/es/css10/vits"

STREAM_URL = f"http://{ESP32_IP}:81/stream"
COMANDO_URL = f"http://{ESP32_IP}:8080/cmd?msg="

TIEMPO_CONFIANZA = 2.5
TIEMPO_COOLDOWN = 3.0
VENTANA = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# MODELO NEURONAL (GESTOS)
# ==============================
class DGRNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

print("Cargando modelo neuronal...")
encoder = joblib.load(ENCODER_PATH)
scaler = joblib.load(SCALER_PATH)
num_classes = len(encoder.classes_)
model = DGRNet(126, num_classes).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Modelo cargado correctamente.\n")

# ==============================
# MEDIAPIPE
# ==============================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# ==============================
# ESTADO GLOBAL
# ==============================
frame_actual = None
captura_activa = True
pred_hist = deque(maxlen=VENTANA)
gesto_en_confianza = None
inicio_confianza = None
ultimo_ejecutado = None
inicio_cooldown = 0
bloqueado = False
modo_super_activo = False

# ==============================
# COMANDOS
# ==============================
comandos = {
    "abrirPuerta": "CMD|OPEN_DOOR",
    "modoSuper": "CMD|SUPER_ON",
    "modoNormal": "CMD|SUPER_OFF",
    "siguienteMelodia": "CMD|CHANGE_MUSIC",
    "anteriorMelodia": "CMD|PREV_MUSIC",
    "encenderLuces": "CMD|LED_ON",
    "apagarLuces": "CMD|LED_OFF"
}

def enviar_comando(cmd):
    def tarea():
        try:
            requests.get(COMANDO_URL + cmd, timeout=1)
            print(f"[ENVÍO] {cmd}")
        except Exception as e:
            print(f"[ERROR envío] {e}")
    threading.Thread(target=tarea, daemon=True).start()

# ==============================
# PHRASE NET AR (Generador de texto)
# ==============================
print("Cargando PhraseNetAR...")

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)
num_cmds = cfg["num_cmds"]
vocab_size = cfg["vocab_size"]
emb_dim = cfg["emb_dim"]
hidden_dim = cfg["hidden_dim"]
max_len = cfg["max_len"]

add_safe_globals([spm.SentencePieceProcessor])
tokenizer = torch.load(TOKENIZER_PATH, weights_only=False)
cmd2id = torch.load(CMD2ID_PATH)
id2cmd = torch.load(ID2CMD_PATH)

class PhraseNetAR(nn.Module):
    def __init__(self, num_cmds, vocab_size, emb_dim, hidden_dim, max_len):
        super().__init__()
        self.emb_cmd = nn.Embedding(num_cmds, emb_dim)
        self.emb_tok = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, cmd_ids, tokens_in):
        seq_len = tokens_in.size(1)
        cmd_emb = self.emb_cmd(cmd_ids).unsqueeze(1).repeat(1, seq_len, 1)
        tok_emb = self.emb_tok(tokens_in)
        pos_ids = torch.arange(seq_len).unsqueeze(0).to(tokens_in.device)
        pos_emb = self.pos_emb(pos_ids)
        x = tok_emb + cmd_emb + pos_emb
        out, _ = self.gru(x)
        return self.fc(out)

model_ar = PhraseNetAR(num_cmds, vocab_size, emb_dim, hidden_dim, max_len).to(DEVICE)
model_ar.load_state_dict(torch.load(MODEL_PHRASE, map_location=DEVICE))
model_ar.eval()

def generar_frase(comando, temperature=0.7, max_tokens=40):
    seq = [tokenizer.bos_id()]
    cmd_tensor = torch.tensor([cmd2id[comando]], device=DEVICE)
    for _ in range(max_tokens):
        seq_tensor = torch.tensor([seq], device=DEVICE)
        logits = model_ar(cmd_tensor, seq_tensor)[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        seq.append(next_id)
        if next_id == tokenizer.eos_id():
            break
    return tokenizer.decode_ids(seq)

# ==============================
# MODELO DE AUDIO VITS
# ==============================
print("Cargando modelo de voz (VITS)...")
tts = TTS(MODEL_VITS)

def reproducir_audio(texto):
    def tarea():
        try:
            wav = tts.tts(texto)
            wav = np.array(wav, dtype=np.float32)
            print(f"[AUDIO] {texto}")
            import sounddevice as sd
            sd.play(wav, 22050)
            sd.wait()
        except Exception as e:
            print(f"[ERROR AUDIO] {e}")
    threading.Thread(target=tarea, daemon=True).start()

# ==============================
# CAPTURA DE VIDEO (HILO)
# ==============================
def hilo_captura():
    global frame_actual, captura_activa
    cap = cv2.VideoCapture(STREAM_URL)
    if not cap.isOpened():
        print("No se pudo conectar con la cámara ESP32.")
        captura_activa = False
        return
    print("Cámara conectada. Reconociendo gestos...")
    while captura_activa:
        ret, frame = cap.read()
        if ret:
            frame_actual = cv2.flip(frame, 1)
        else:
            time.sleep(0.05)
    cap.release()

threading.Thread(target=hilo_captura, daemon=True).start()

# ==============================
# LOOP PRINCIPAL
# ==============================
try:
    while True:
        if frame_actual is None:
            time.sleep(0.05)
            continue

        frame = frame_actual.copy()
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        puntos = []

        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                for lm in hand_landmarks.landmark:
                    puntos.extend([lm.x, lm.y, lm.z])
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            while len(puntos) < 126:
                puntos.append(0.0)
            if len(puntos) > 126:
                puntos = puntos[:126]

            X = scaler.transform([puntos])
            X_tensor = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            with torch.no_grad():
                logits = model(X_tensor)
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                pred = encoder.inverse_transform([np.argmax(probs)])[0]
            pred_hist.append(pred)
        else:
            pred_hist.clear()
            pred = None

        pred_suavizado = max(set(pred_hist), key=pred_hist.count) if pred_hist else None
        ahora = time.time()

        # === CONFIRMACIÓN Y CONTROL ===
        if not bloqueado and pred_suavizado and pred_suavizado in comandos:
            if modo_super_activo and pred_suavizado in ["encenderLuces", "apagarLuces"]:
                pass
            else:
                if gesto_en_confianza != pred_suavizado:
                    gesto_en_confianza = pred_suavizado
                    inicio_confianza = ahora
                else:
                    duracion = ahora - inicio_confianza
                    if duracion >= TIEMPO_CONFIANZA:
                        cmd = comandos[gesto_en_confianza]
                        enviar_comando(cmd)

                        # === Integración del generador de frase y audio ===
                        frase = generar_frase(gesto_en_confianza)
                        print(f"[PHRASE NET] '{gesto_en_confianza}' → {frase}")
                        reproducir_audio(frase)

                        ultimo_ejecutado = gesto_en_confianza
                        bloqueado = True
                        inicio_cooldown = ahora

                        if gesto_en_confianza == "modoSuper":
                            modo_super_activo = True
                            print(">> Modo SUPER activado: luces bloqueadas.")
                        elif gesto_en_confianza == "modoNormal":
                            modo_super_activo = False
                            print(">> Modo NORMAL activado: luces desbloqueadas.")

                        gesto_en_confianza = None
                        inicio_confianza = None
        elif bloqueado:
            if (ahora - inicio_cooldown) >= TIEMPO_COOLDOWN:
                bloqueado = False
        else:
            gesto_en_confianza = None
            inicio_confianza = None

        # === VISUAL ===
        texto = ""
        color = (0, 255, 0)
        if gesto_en_confianza:
            restante = TIEMPO_CONFIANZA - (ahora - inicio_confianza)
            texto = f"Detectando {gesto_en_confianza} ({restante:.1f}s)"
        elif bloqueado:
            restante = TIEMPO_COOLDOWN - (ahora - inicio_cooldown)
            texto = f"Ejecutado '{ultimo_ejecutado}' Esperando {restante:.1f}s"
            color = (0, 140, 255)
        elif pred_suavizado:
            texto = f"Gesto: {pred_suavizado} | ModoSuper: {'ON' if modo_super_activo else 'OFF'}"

        if texto:
            cv2.putText(frame, texto, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        cv2.imshow("Reconocimiento de gestos - SynkroHogar", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("\nCerrando sistema SynkroHogar IA...")
            captura_activa = False
            cv2.destroyAllWindows()
            sys.exit(0)

except KeyboardInterrupt:
    print("\nCierre manual detectado.")
    captura_activa = False
    cv2.destroyAllWindows()
    sys.exit(0)


a:\GeneracionImagenes\finalOmega\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Cargando modelo neuronal...
Modelo cargado correctamente.

Cargando PhraseNetAR...
Cargando modelo de voz (VITS)...
 > tts_models/es/css10/vits is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > initialization of speaker-embedding layers.
 > initialization of

SystemExit: 0

a:\GeneracionImagenes\finalOmega\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
